# Milestone - Unit 5: geometric steering

Unit 5 asked whether a controller that **looks ahead** can survive what a controller that only reacts cannot: an actuator delay at speed.
`agents/unit-05.py` is Unit 4's file with three steering laws behind one constant, `STEERING`:
`"pursuit"` (the default: pure pursuit with the lookahead L_d = clip(`PP_K`·v, 3, 40) m, `PP_K` = 0.3 s),
`"pid"` (Unit 4's law exactly) and `"stanley"`.
Its PI speed loop, instruments and model are Unit 4's.

The notebook asserts the unit's gates (plan `roar-005`), the multi-seed ones on the trimmed subset the plan names; a routed racekit test (`tests/test_racekit_unit04_05_premise.py`) races every five-seed figure in full:

- **U5.1** pure pursuit on a circle: the submission's `pursuit_curvature` returns 1/R and `pursuit_wheel_angle` returns atan(L/R), to 1e-12, for R ∈ {18, 25, 120} m and chords d ∈ {3, 6, 9, 12} m;
- **U5.2** the submission, deterministic (jitter 0, no delay): 216.35 / 400.50 / 858.45 s, clean, RMS 0.0035 / 0.0193 / 0.0157 m; `check_submission` passes; with `STEERING = "pid"` it races exactly like `agents/unit-04.py`, to the tick;
- **U5.3** the 20 m/s scorecard (jitter 0.05, seeds 0–4, no delay): pure pursuit's and Stanley's RMS medians, all clean, and pure pursuit does **not** beat Unit 4's medians (U4.3);
- **U5.4** the delay study (`test-circuit`, 30 m/s, jitter 0.05, 400 s cap): at lag 0 all four controllers are clean on seeds 0–4 and Unit 4's PD beats pure pursuit beyond both ranges; at lag 2 pure pursuit is clean on seeds 0–4, Unit 4's PD collides on every seed, the detuned PD is clean and pure pursuit beats it beyond both ranges; Stanley and the starter at lag 2 on seed 0;
- **U5.5** the lookahead sweep on seed 0: the best `PP_K` moves from 0.2 s at lag 0 to 0.3 s at lag 2.

Tolerances (the plan's Gates header): deterministic and median race times within one tick (±0.05 s); RMS |offset| (medians and ranges) within ±0.0005 m; outcomes and per-seed collision lists exact; every noise-aware comparison strict.
Every figure is a model number on racekit's **kinematic** car.

## 1. Set-up

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.
A variant is the reference file with some constants changed (`write_variant`), written to a temporary directory.

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.check_submission import check_submission
from racekit.measure import repeat_races
from racekit.runner import load_submission, run_race, shim_installed

AGENTS = Path("..") / "agents"
AGENT = AGENTS / "unit-05.py"
UNIT_04 = AGENTS / "unit-04.py"
STARTER = AGENTS / "unit-01.py"
TRACKS = ("oval", "test-circuit", "monza-like")
SEEDS = (0, 1, 2, 3, 4)
TICK = 0.05 + 1e-9
RMS_TOL = 0.0005 + 1e-9
WHEELBASE = 2.875
workdir = Path(mkdtemp(prefix="roar-milestone-05-"))


def variant(source, name, **constants):
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def rms_offset(result):
    telemetry = result.telemetry
    offset = telemetry["offset"][telemetry["held"] == 0]
    return float(np.sqrt(np.mean(offset ** 2)))


def clean(result):
    return result.finished and result.collisions == 0


def rms_stats(results):
    # (median, range) of the per-seed RMS cross-track errors
    values = [rms_offset(result) for result in results]
    return float(np.median(values)), max(values) - min(values)


def time_stats(results):
    # (median, spread) of the per-seed race times
    values = [result.total_time for result in results]
    return float(np.median(values)), max(values) - min(values)


def seeds_race(source, track, lag=0, seeds=SEEDS, max_seconds=1500.0):
    def car_factory():
        return KinematicCar(jitter=0.05, lag_ticks=lag)

    return repeat_races(source, track, n=len(seeds), seeds=seeds, car_factory=car_factory,
                        max_seconds=max_seconds).results

## 2. U5.1: pure pursuit on a circle

A car on a circle of radius R, tangent to it, and a target on the circle at chord distance d: the tangent–chord angle gives sin α = d / 2R, so κ = 2 sin α / d = 1/R for every d, and the wheel angle is atan(L/R).
The target is built here from the circle's centre and its central angle, independently of the submission, and seen from the car through the car frame.

In [ ]:
with shim_installed():
    Agent, module_name = load_submission(AGENT)
submission = sys.modules[module_name]


def to_car_frame(px, py, cx, cy, yaw):
    dx, dy = px - cx, py - cy
    return math.cos(yaw) * dx + math.sin(yaw) * dy, -math.sin(yaw) * dx + math.cos(yaw) * dy


worst_kappa = worst_delta = 0.0
car_x, car_y, car_yaw = -12.0, 30.0, 2.3
for radius in (18.0, 25.0, 120.0):
    centre_x, centre_y = car_x - radius * math.sin(car_yaw), car_y + radius * math.cos(car_yaw)
    for chord in (3.0, 6.0, 9.0, 12.0):
        theta = 2.0 * math.atan2(chord / 2.0, math.sqrt(radius * radius - chord * chord / 4.0))  # central angle
        tx = centre_x + radius * math.sin(car_yaw + theta)
        ty = centre_y - radius * math.cos(car_yaw + theta)
        forward, left = to_car_frame(tx, ty, car_x, car_y, car_yaw)
        alpha, distance = math.atan2(left, forward), math.hypot(forward, left)
        assert abs(distance - chord) < 1e-12
        worst_kappa = max(worst_kappa, abs(submission.pursuit_curvature(alpha, distance) - 1.0 / radius))
        worst_delta = max(worst_delta, abs(submission.pursuit_wheel_angle(alpha, distance)
                                           - math.atan2(WHEELBASE / radius, 1.0)))
print(f"worst |kappa - 1/R| = {worst_kappa:.1e} 1/m, worst |delta - atan(L/R)| = {worst_delta:.1e} rad")
assert worst_kappa < 1e-12 and worst_delta < 1e-12

## 3. U5.2: the submission, deterministic

No jitter and no delay, on all three tracks.
Then the equivalence: the same file with `STEERING = "pid"` must race exactly like `agents/unit-04.py`, so its telemetry is compared column by column, and Unit 4's pinned U4.2 times and RMS hold for it.

In [ ]:
PINNED = {"oval": (216.35, 0.0035), "test-circuit": (400.50, 0.0193), "monza-like": (858.45, 0.0157)}
UNIT_04_PINNED = {"oval": (216.35, 0.0034), "test-circuit": (400.55, 0.0139), "monza-like": (858.45, 0.0112)}
pid_variant = variant(AGENT, "unit-05-pid", STEERING="pid")
for track in TRACKS:
    result = run_race(AGENT, track, car=KinematicCar(jitter=0.0))
    time, rms = PINNED[track]
    print(f"{track:>12}: pure pursuit {result.total_time:.2f} s, RMS {rms_offset(result):.4f} m, "
          f"{result.collisions} collisions")
    assert clean(result) and abs(result.total_time - time) <= TICK and abs(rms_offset(result) - rms) <= RMS_TOL

    as_pid = run_race(pid_variant, track, car=KinematicCar(jitter=0.0))
    unit_04 = run_race(UNIT_04, track, car=KinematicCar(jitter=0.0))
    assert len(as_pid.telemetry) == len(unit_04.telemetry)
    for column in unit_04.telemetry.dtype.names:  # identical telemetry, column by column
        assert np.max(np.abs(as_pid.telemetry[column] - unit_04.telemetry[column])) == 0, (track, column)
    time, rms = UNIT_04_PINNED[track]
    assert clean(as_pid) and abs(as_pid.total_time - time) <= TICK and abs(rms_offset(as_pid) - rms) <= RMS_TOL
    print(f"{'':>12}  STEERING = 'pid': {as_pid.total_time:.2f} s, RMS {rms_offset(as_pid):.4f} m, "
          "identical to agents/unit-04.py")

report = check_submission(AGENT)
print(report)
assert report.ok, report.findings

## 4. U5.3: the 20 m/s scorecard

Jitter 0.05, seeds 0–4, no delay: pure pursuit and Stanley on every track.
Unit 4's medians are its own milestone's U4.3 pins.

In [ ]:
UNIT_04_MEDIANS = {"oval": 0.1041, "test-circuit": 0.1048, "monza-like": 0.1029}  # U4.3
PURSUIT_20 = {"oval": (0.1422, 0.0107), "test-circuit": (0.1481, 0.0050), "monza-like": (0.1445, 0.0022)}
scorecard = {}
for track in TRACKS:
    scorecard[("pursuit", track)] = seeds_race(AGENT, track)
    median, spread = rms_stats(scorecard[("pursuit", track)])
    print(f"{track:>12}: pure pursuit RMS median {median:.4f} m (range {spread:.4f}); Unit 4 {UNIT_04_MEDIANS[track]:.4f}")
    assert all(clean(result) for result in scorecard[("pursuit", track)])
    assert abs(median - PURSUIT_20[track][0]) <= RMS_TOL and abs(spread - PURSUIT_20[track][1]) <= RMS_TOL
    assert median > UNIT_04_MEDIANS[track]  # pure pursuit does not beat Unit 4 without a delay

In [ ]:
STANLEY_20 = {"oval": 0.0987, "test-circuit": 0.1043, "monza-like": 0.0981}
stanley_variant = variant(AGENT, "unit-05-stanley", STEERING="stanley")
for track in TRACKS:
    scorecard[("stanley", track)] = seeds_race(stanley_variant, track)
    median, spread = rms_stats(scorecard[("stanley", track)])
    print(f"{track:>12}: Stanley RMS median {median:.4f} m (range {spread:.4f})")
    assert all(clean(result) for result in scorecard[("stanley", track)])
    assert abs(median - STANLEY_20[track]) <= RMS_TOL

## 5. U5.4: the delay study

`test-circuit`, `TARGET_SPEED = 30.0`, jitter 0.05, `max_seconds=400`, on `KinematicCar(jitter=0.05, lag_ticks=N)`.
The starter keeps its P speed loop; the other controllers use Unit 4's PI.

In [ ]:
FAST = {"TARGET_SPEED": 30.0}
CONTROLLERS = {  # name -> the variant at 30 m/s
    "pd": variant(UNIT_04, "pd-30", **FAST),
    "stanley": variant(AGENT, "stanley-30", STEERING="stanley", **FAST),
    "pursuit": variant(AGENT, "pursuit-30", **FAST),
    "starter": variant(STARTER, "starter-30", **FAST),
    "detuned": variant(UNIT_04, "detuned-30", K_E=0.05, K_HEAD=0.5, **FAST),
}
delay = {}
LAG_0 = {"pd": (0.1295, 0.0060), "stanley": (0.1582, 0.0139), "pursuit": (0.3097, 0.0153),
         "starter": (0.2346, 0.0167)}
for name, (median_pin, range_pin) in LAG_0.items():
    delay[(name, 0)] = seeds_race(CONTROLLERS[name], "test-circuit", lag=0, max_seconds=400.0)
    median, spread = rms_stats(delay[(name, 0)])
    print(f"lag 0, {name:>8}: collisions {[r.collisions for r in delay[(name, 0)]]}, RMS median {median:.4f} m (range {spread:.4f})")
    assert [result.collisions for result in delay[(name, 0)]] == [0, 0, 0, 0, 0]
    assert all(clean(result) for result in delay[(name, 0)])
    assert abs(median - median_pin) <= RMS_TOL and abs(spread - range_pin) <= RMS_TOL

In [ ]:
# lag 2, five seeds: Unit 4's PD (it crashes, so each race runs to the 400 s cap)
delay[("pd", 2)] = seeds_race(CONTROLLERS["pd"], "test-circuit", lag=2, max_seconds=400.0)
collisions = [result.collisions for result in delay[("pd", 2)]]
print("lag 2, Unit 4 PD: collisions per seed", collisions)
assert collisions == [41, 42, 43, 43, 40]
assert not any(result.finished for result in delay[("pd", 2)])

In [ ]:
# lag 2, five seeds: pure pursuit and the detuned PD (both clean)
LAG_2_CLEAN = {"pursuit": (0.4526, 0.0234, 269.70, 0.20), "detuned": (0.5149, 0.0376, 269.70, 0.15)}
for name, (median_pin, range_pin, time_pin, spread_pin) in LAG_2_CLEAN.items():
    delay[(name, 2)] = seeds_race(CONTROLLERS[name], "test-circuit", lag=2, max_seconds=400.0)
    median, spread = rms_stats(delay[(name, 2)])
    time_median, time_spread = time_stats(delay[(name, 2)])
    print(f"lag 2, {name:>8}: clean {sum(clean(r) for r in delay[(name, 2)])}/5, RMS median {median:.4f} m "
          f"(range {spread:.4f}), time median {time_median:.2f} s (spread {time_spread:.2f})")
    assert all(clean(result) for result in delay[(name, 2)])
    assert abs(median - median_pin) <= RMS_TOL and abs(spread - range_pin) <= RMS_TOL
    assert abs(time_median - time_pin) <= TICK and abs(time_spread - spread_pin) <= TICK

# lag 2, seed 0 only (the runtime trim): Stanley crashes; the starter does not finish inside the cap
stanley_2 = run_race(CONTROLLERS["stanley"], "test-circuit", car=KinematicCar(jitter=0.05, lag_ticks=2),
                     seed=0, max_seconds=400.0)
starter_2 = run_race(CONTROLLERS["starter"], "test-circuit", car=KinematicCar(jitter=0.05, lag_ticks=2),
                     seed=0, max_seconds=400.0)
print(f"lag 2, seed 0: Stanley {stanley_2.collisions} collisions; starter {starter_2.collisions} collisions, "
      f"finished {starter_2.finished}")
assert stanley_2.collisions == 20 and not stanley_2.finished
assert starter_2.collisions == 0 and not starter_2.finished

### The gate

Stability against corner-cutting, both halves, noise-aware:

In [ ]:
# at lag 2, pure pursuit beats the detuned PD by more than both ranges ...
(p_median, p_range), (d_median, d_range) = rms_stats(delay[("pursuit", 2)]), rms_stats(delay[("detuned", 2)])
assert d_median - p_median > max(p_range, d_range)
# ... the tight laws collide (Unit 4's PD on every seed, Stanley on seed 0) and pure pursuit is clean 5/5 ...
assert all(result.collisions > 0 for result in delay[("pd", 2)]) and stanley_2.collisions > 0
assert all(clean(result) for result in delay[("pursuit", 2)])
# ... and at lag 0 Unit 4's PD beats pure pursuit by more than both ranges (the corner-cutting price)
(pd_median, pd_range), (p0_median, p0_range) = rms_stats(delay[("pd", 0)]), rms_stats(delay[("pursuit", 0)])
assert p0_median - pd_median > max(pd_range, p0_range)
print(f"lag 2: pure pursuit {p_median:.4f} m vs detuned PD {d_median:.4f} m (gap {d_median - p_median:.4f} > {max(p_range, d_range):.4f})")
print(f"lag 0: Unit 4 PD {pd_median:.4f} m vs pure pursuit {p0_median:.4f} m (gap {p0_median - pd_median:.4f} > {max(pd_range, p0_range):.4f})")

## 6. U5.5: the lookahead sweep

`PP_K` ∈ {0.2, 0.3, 0.4, 0.5} s at lags 0 and 2, seed 0 (the five-seed medians are the routed test's).
`PP_K = 0.3` is the delay study's seed-0 pure-pursuit race.

In [ ]:
SWEEP_PINS = {  # (PP_K, lag): (time, RMS) on seed 0
    (0.2, 0): (268.70, 0.1823), (0.3, 0): (268.75, 0.3097), (0.4, 0): (268.80, 0.4631), (0.5, 0): (268.75, 0.6477),
    (0.2, 2): (294.35, 1.3479), (0.3, 2): (269.70, 0.4559), (0.4, 2): (269.50, 0.5677), (0.5, 2): (269.45, 0.7203),
}
sweep = {}
for (k, lag), (time, rms) in SWEEP_PINS.items():
    if k == 0.3:
        result = delay[("pursuit", lag)][0]  # seed 0
    else:
        path = variant(AGENT, f"pursuit-30-k{k}", PP_K=k, **FAST)
        result = run_race(path, "test-circuit", car=KinematicCar(jitter=0.05, lag_ticks=lag), seed=0, max_seconds=400.0)
    sweep[(k, lag)] = rms_offset(result)
    print(f"PP_K = {k} s, lag {lag}: {result.total_time:.2f} s, RMS {sweep[(k, lag)]:.4f} m")
    assert clean(result) and abs(result.total_time - time) <= TICK and abs(sweep[(k, lag)] - rms) <= RMS_TOL

best = {lag: min((0.2, 0.3, 0.4, 0.5), key=lambda k: sweep[(k, lag)]) for lag in (0, 2)}
print("the best lookahead per lag:", best)
assert best == {0: 0.2, 2: 0.3}  # the best lookahead grows with the delay

## Unit 5, pinned

| | no delay | 2-tick delay, 30 m/s |
|---|---|---|
| Unit 4's PD | best line (0.10 m at 20 m/s, 0.13 m at 30 m/s) | crashes on every seed |
| Stanley | about as good as the PD | crashes |
| detuned PD (K_E 0.05, K_HEAD 0.5) | clean, well off the line | clean, 0.51 m |
| **pure pursuit, `PP_K` = 0.3 s** | cuts corners (0.14 m at 20 m/s, 0.31 m at 30 m/s) | **clean, 0.45 m** |

The submission ships pure pursuit, the robust choice while the real car's delay is unknown, with Unit 4's law one constant away (`STEERING = "pid"`, identical to the tick).
The Unit 5 lab runs both on CARLA; Unit 8 measures the delay.